In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:
nopes_clean_file = Path(project_root / "data/people/ nopes_clean.json")
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")
matches_file = Path(project_root / "data/people/matches_merged.json")

In [3]:
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)

with open(nopes_clean_file, "r") as f:
   nopes = json.load(f)


In [4]:
matches_found = []
new_people = []

# for person in new_people[:100]:
for person in nopes:
    if person["match_found"]:
        matches_found.append(person)
    else:
        new_people.append(person)


In [5]:
with open("matches_from_nopes.json", "r") as f:
   matched_new = json.load(f)
with open(matches_file, "r") as f:
   matches_dict = json.load(f)


unmatched_dict = {}
# rprint(f"unmatched entries: {len(unmatched)}")
# total_unmatched_flat = len(unmatched)
# total_nopes = len(nopes)

for entry in unmatched:
    composite_id = entry["composite_id"]

    if composite_id not in unmatched_dict:
        unmatched_dict[composite_id] = [entry]
    else:
        unmatched_dict[composite_id].append(entry)

rprint(dict(list(unmatched_dict.items())[:1]))
rprint(matched_new[:1])
rprint(dict(list(matches_dict.items())[:1]))

{
    'wien_210_9_10': [
        {
            'composite_id': 'wien_210_9_10',
            'display_name': 'ROMMEL, Otto',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False,
            'is_organisation': False,
            'display_norm': 'rommel, otto',
            'last': 'ROMMEL',
            'first': 'Otto',
            'single': None,
            'original_entry': 'ROMMEL, Otto\nEin Jahrhundert Alt-Wiener Parodie. Herausgegeben von … Wien 
Österreichischer Bundesverlag 1930. 285 S. OLn.',
            'verification_notes': "The entry states 'Herausgegeben von …' but no editor name follows the ellipsis. 
missing_person flag set accordingly. Note: ROMMEL, Otto is listed as the main entry (author/compiler), but the 
'Herausgegeben von …' suggests an editor role may be missing or that Rommel himself is the editor with the name 
omitted by the cataloger.",
            'match_found': False,
            'person_id': None
        }
    ]
}

[
    {
        'display_norm': 'stendhal, friedrich',
        'composite_id': ['fremdsprachige_857_35_38'],
        'family_name': None,
        'given_names': None,
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': 'Stendhal',
        'is_organisation': False,
        'match_found': True,
        'matched_unified_id': 'stendhal',
        'matched_person_id': 6717,
        'unified_id': None,
        '_source_custom_id': 'batch_nopes_035'
    }
]

{
    '5315': [
        {
            'display_name': 'PLAUTUS, T.',
            'composite_id': 'antike_109_5_9',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False
        }
    ]
}

In [6]:
rprint(f"total entries: {len(matched_new)}")

id_in_matches_dict = 0
not_in_matches_dict = 0

for person in matched_new:
    name = person["display_norm"]
    person_id = str(person["matched_person_id"])
    composite_ids = person["composite_id"]
    for composite_id in composite_ids:
        records = unmatched_dict[composite_id]
        found = False

        for record in records:
            if record["display_norm"] == name:
                found = True
                display_name = record["display_name"]
                sort_order = record["sort_order"]
                is_author = record["is_author"]
                is_editor = record["is_editor"]
                is_contributor = record["is_contributor"]
                is_translator = record["is_translator"]

                if person_id not in matches_dict:
                    not_in_matches_dict += 1
                    matches_dict[person_id] = [{
                        "display_name": display_name,
                        "composite_id": composite_id,
                        "sort_order": sort_order,
                        "is_author": is_author,
                        "is_editor": is_editor,
                        "is_contributor": is_contributor,
                        "is_translator": is_translator
                    }]
                else:
                    id_in_matches_dict += 1
                    existing = [e["composite_id"] for e in matches_dict[person_id]]
                    if composite_id not in existing:
                        matches_dict[person_id].append({
                            "display_name": display_name,
                            "composite_id": composite_id,
                            "sort_order": sort_order,
                            "is_author": is_author,
                            "is_editor": is_editor,
                            "is_contributor": is_contributor,
                            "is_translator": is_translator
                        })
                    break
        if not found:
            rprint(f"no match: {composite_id} | looking for: {name}")
            rprint(f"  records have: {[r['display_norm'] for r in records]}")

with open("latest_matches_merged.json", "w") as f:
    json.dump(matches_dict, f, ensure_ascii=False, indent=2)

rprint(f"new id: {not_in_matches_dict}")
rprint(f"old id: {id_in_matches_dict}")

total entries: 42

new id: 24

old id: 21

In [7]:
matched_norms = {person["display_norm"] for person in matched_new}
now_matched = [person for person in new_people if person["display_norm"] in matched_norms]
rprint(f"previously unmatched, now matched: {len(now_matched)}")

previously unmatched, now matched: 0